# Fed Chair Transition Dynamics

Three event windows per transition:
- **Pre-announcement**: 90 days before nomination
- **Limbo**: nomination date → start date
- **Post-start**: 90 days after start date

Measures:
1. Spot yields at each tenor (indexed to 0 at nomination)
2. Key spreads: 2s10s, 3m10y
3. Realised vol: 21-day rolling σ of daily yield changes
4. Rolling PCA components (level / slope / curvature)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

try:
    import pandas_datareader.data as web
    HAS_PDR = True
except ImportError:
    HAS_PDR = False
    print('pandas_datareader not found — pip install pandas-datareader')

plt.rcParams.update({
    'figure.facecolor': '#0d1117',
    'axes.facecolor': '#161b22',
    'axes.edgecolor': '#30363d',
    'text.color': '#e6edf3',
    'axes.labelcolor': '#e6edf3',
    'xtick.color': '#8b949e',
    'ytick.color': '#8b949e',
    'grid.color': '#21262d',
    'grid.linestyle': '--',
    'grid.alpha': 0.5,
    'axes.titlecolor': '#e6edf3',
    'legend.facecolor': '#161b22',
    'legend.edgecolor': '#30363d',
    'font.size': 10,
})

In [ ]:
# ── Fed Chair transition dates ───────────────────────────────────────────────
# nomination_date = public announcement (the event)
# start_date      = first day in office
# Update 'powell_2026' once the nominee and dates are confirmed

TRANSITIONS = [
    dict(label='Bernanke',      chair='Ben Bernanke',
         nomination='2005-10-24', confirm='2006-01-31', start='2006-02-01'),
    dict(label='Yellen',        chair='Janet Yellen',
         nomination='2013-10-09', confirm='2014-01-06', start='2014-02-03'),
    dict(label='Powell (1st)',  chair='Jerome Powell',
         nomination='2017-11-02', confirm='2018-01-23', start='2018-02-05'),
    dict(label='Powell (2nd)',  chair='Jerome Powell (renominated)',
         nomination='2021-11-22', confirm='2022-05-23', start='2022-05-23'),
    # Powell term ends 2026-05-15; update once successor is confirmed
    # dict(label='[Successor]', chair='???',
    #      nomination='2025-??-??', confirm='2026-??-??', start='2026-05-15'),
]

for t in TRANSITIONS:
    t['nomination'] = pd.Timestamp(t['nomination'])
    t['confirm']    = pd.Timestamp(t['confirm'])
    t['start']      = pd.Timestamp(t['start'])
    t['limbo_days'] = (t['start'] - t['nomination']).days

pd.DataFrame(TRANSITIONS)[['label','chair','nomination','confirm','start','limbo_days']]

In [ ]:
# ── Fetch Treasury yields from FRED ─────────────────────────────────────────
FRED_SERIES = {
    '1M':  'DGS1MO',
    '3M':  'DGS3MO',
    '6M':  'DGS6MO',
    '1Y':  'DGS1',
    '2Y':  'DGS2',
    '3Y':  'DGS3',
    '5Y':  'DGS5',
    '7Y':  'DGS7',
    '10Y': 'DGS10',
    '20Y': 'DGS20',
    '30Y': 'DGS30',
}

START_FETCH = '2004-01-01'
END_FETCH   = '2026-05-03'

assert HAS_PDR, 'Install pandas_datareader first'

raw = web.DataReader(list(FRED_SERIES.values()), 'fred', START_FETCH, END_FETCH)
raw.columns = list(FRED_SERIES.keys())
raw = raw.ffill().dropna()

print(f'Loaded {len(raw)} days  {raw.index[0].date()} → {raw.index[-1].date()}')
raw.tail(3)

In [ ]:
# ── Helpers ──────────────────────────────────────────────────────────────────
TENORS      = list(FRED_SERIES.keys())
TENOR_MATS  = [1/12, 3/12, 6/12, 1, 2, 3, 5, 7, 10, 20, 30]  # years, for PCA axis

PRE_DAYS  = 90
POST_DAYS = 90

PALETTE = [
    '#58a6ff', '#3fb950', '#d29922', '#f78166', '#bc8cff',
    '#79c0ff', '#56d364', '#e3b341', '#ffa198', '#d2a8ff',
]

def window_data(df: pd.DataFrame, t: dict, pre=PRE_DAYS, post=POST_DAYS):
    """Slice df to [nomination - pre, start + post]."""
    lo = t['nomination'] - pd.Timedelta(days=pre)
    hi = t['start']      + pd.Timedelta(days=post)
    return df.loc[lo:hi].copy()

def days_from_nomination(df: pd.DataFrame, nomination: pd.Timestamp) -> pd.Series:
    return (df.index - nomination).days

---
## 1 · Spot Yields — level changes indexed to nomination

In [ ]:
PLOT_TENORS = ['2Y', '5Y', '10Y', '30Y']  # fewer lines = clearer chart

fig, axes = plt.subplots(2, 2, figsize=(16, 10), sharey=False)
axes = axes.flatten()

for ax, t in zip(axes, TRANSITIONS):
    wd = window_data(raw, t)
    nom = t['nomination']
    base = raw.loc[:nom].iloc[-1]           # level on nomination day
    indexed = wd[PLOT_TENORS] - base[PLOT_TENORS]
    x = days_from_nomination(indexed, nom)

    for i, tenor in enumerate(PLOT_TENORS):
        ax.plot(x, indexed[tenor], color=PALETTE[i], lw=1.6, label=tenor)

    # shade limbo period
    limbo_end_days = (t['start'] - nom).days
    ax.axvspan(0, limbo_end_days, alpha=0.12, color='#d29922', label='Limbo')
    ax.axvline(0,             color='#d29922', lw=1.2, ls='--', alpha=0.8)
    ax.axvline(limbo_end_days, color='#3fb950', lw=1.2, ls='--', alpha=0.8)
    ax.axhline(0, color='#8b949e', lw=0.7, ls=':')

    ax.set_title(f'{t["label"]}  ({t["nomination"].strftime("%b %Y")} → {t["start"].strftime("%b %Y")})')
    ax.set_xlabel('Calendar days from nomination')
    ax.set_ylabel('Δ yield (bps relative to nomination day)')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v*100:.0f}'))
    ax.legend(fontsize=8, loc='upper left')
    ax.grid(True)

fig.suptitle('Spot yield changes around Fed Chair transitions\n(nomination = day 0; green dashes = start date)', y=1.01)
plt.tight_layout()
plt.show()

---
## 2 · Key Spreads: 2s10s and 3m10y

In [ ]:
raw['2s10s'] = raw['10Y'] - raw['2Y']
raw['3m10y'] = raw['10Y'] - raw['3M']

SPREADS = ['2s10s', '3m10y']

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, t in zip(axes, TRANSITIONS):
    wd = window_data(raw, t)[SPREADS]
    nom = t['nomination']
    x   = days_from_nomination(wd, nom)

    ax.plot(x, wd['2s10s'] * 100, color='#58a6ff', lw=1.6, label='2s10s')
    ax.plot(x, wd['3m10y'] * 100, color='#f78166', lw=1.6, label='3m10y')

    limbo_days = (t['start'] - nom).days
    ax.axvspan(0, limbo_days, alpha=0.12, color='#d29922')
    ax.axvline(0,          color='#d29922', lw=1.2, ls='--', alpha=0.8)
    ax.axvline(limbo_days, color='#3fb950', lw=1.2, ls='--', alpha=0.8)
    ax.axhline(0, color='#8b949e', lw=0.7, ls=':')

    ax.set_title(t['label'])
    ax.set_xlabel('Days from nomination')
    ax.set_ylabel('Spread (bps)')
    ax.legend(fontsize=8)
    ax.grid(True)

fig.suptitle('Curve spreads around Fed Chair transitions', y=1.01)
plt.tight_layout()
plt.show()

---
## 3 · Realised Volatility — 21-day rolling σ of daily yield changes

In [ ]:
RVOL_TENORS = ['2Y', '5Y', '10Y', '30Y']

daily_chg = raw[TENORS].diff() * 100  # in bps
rvol = daily_chg.rolling(21).std()

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

for ax, t in zip(axes, TRANSITIONS):
    lo  = t['nomination'] - pd.Timedelta(days=PRE_DAYS)
    hi  = t['start']      + pd.Timedelta(days=POST_DAYS)
    wd  = rvol.loc[lo:hi, RVOL_TENORS]
    nom = t['nomination']
    x   = days_from_nomination(wd, nom)

    for i, tenor in enumerate(RVOL_TENORS):
        ax.plot(x, wd[tenor], color=PALETTE[i], lw=1.5, label=tenor)

    limbo_days = (t['start'] - nom).days
    ax.axvspan(0, limbo_days, alpha=0.12, color='#d29922')
    ax.axvline(0,          color='#d29922', lw=1.2, ls='--', alpha=0.8)
    ax.axvline(limbo_days, color='#3fb950', lw=1.2, ls='--', alpha=0.8)

    ax.set_title(t['label'])
    ax.set_xlabel('Days from nomination')
    ax.set_ylabel('21d realised vol (bps/day)')
    ax.legend(fontsize=8)
    ax.grid(True)

fig.suptitle('Realised vol (21d rolling σ of daily Δyield) around Fed Chair transitions', y=1.01)
plt.tight_layout()
plt.show()

---
## 4 · PCA on the Yield Curve

**Fitting approach:**  
PCA is fit on the **full train sample pre-nomination** for each event, so the component directions are stable and comparable across windows. We then project the event-window yields onto those directions to get PC scores.

- **PC1** ≈ level (all tenors move together)
- **PC2** ≈ slope (short vs long rates)
- **PC3** ≈ curvature (belly vs wings)

In [ ]:
PCA_TENORS = ['3M', '2Y', '5Y', '10Y', '30Y']   # 5-point curve

fig, axes = plt.subplots(4, 3, figsize=(18, 20))

for row, t in enumerate(TRANSITIONS):
    nom = t['nomination']

    # Fit PCA on the 2 years before nomination
    fit_start = nom - pd.Timedelta(days=730)
    fit_data  = raw.loc[fit_start:nom, PCA_TENORS].dropna()
    scaler    = StandardScaler()
    fit_scaled = scaler.fit_transform(fit_data)
    pca       = PCA(n_components=3)
    pca.fit(fit_scaled)

    evr = pca.explained_variance_ratio_

    # Project event window
    lo   = nom - pd.Timedelta(days=PRE_DAYS)
    hi   = t['start'] + pd.Timedelta(days=POST_DAYS)
    wd   = raw.loc[lo:hi, PCA_TENORS].dropna()
    proj = pca.transform(scaler.transform(wd))
    x    = days_from_nomination(wd, nom)

    pc_labels = [
        f'PC1 level ({evr[0]*100:.1f}%)',
        f'PC2 slope ({evr[1]*100:.1f}%)',
        f'PC3 curv. ({evr[2]*100:.1f}%)',
    ]
    colors = ['#58a6ff', '#3fb950', '#d29922']

    limbo_days = (t['start'] - nom).days

    # Loadings plot
    ax_load = axes[row, 0]
    for j in range(3):
        ax_load.plot(TENOR_MATS[[0,1,2,3,4]], pca.components_[j],
                     color=colors[j], marker='o', ms=4, label=pc_labels[j])
    ax_load.set_title(f'{t["label"]} — PC loadings')
    ax_load.set_xlabel('Tenor (years)')
    ax_load.set_ylabel('Loading')
    ax_load.axhline(0, color='#8b949e', lw=0.7, ls=':')
    ax_load.legend(fontsize=7)
    ax_load.grid(True)

    # PC1 and PC2 scores over event window
    for col, pc_idx in enumerate([0, 1], start=1):
        ax = axes[row, col]
        ax.plot(x, proj[:, pc_idx], color=colors[pc_idx], lw=1.5)
        ax.axvspan(0, limbo_days, alpha=0.12, color='#d29922')
        ax.axvline(0,          color='#d29922', lw=1.2, ls='--', alpha=0.8)
        ax.axvline(limbo_days, color='#3fb950', lw=1.2, ls='--', alpha=0.8)
        ax.axhline(0, color='#8b949e', lw=0.7, ls=':')
        ax.set_title(f'{t["label"]} — {pc_labels[pc_idx]}')
        ax.set_xlabel('Days from nomination')
        ax.set_ylabel('Score (standardised units)')
        ax.grid(True)

fig.suptitle(
    'PCA of yield curve around Fed Chair transitions\n'
    'PCA fit on 2 years pre-nomination  |  shaded = limbo  |  green dash = start date',
    y=1.01
)
plt.tight_layout()
plt.show()

---
## 5 · Rolling PCA — track how loadings shift over time

Rather than fitting once pre-nomination, fit PCA on a 250-day rolling window through each event. This shows whether the *structure* of the curve changes (loadings drift) or whether only the scores move.

In [ ]:
ROLL_WIN = 250  # trading days

def rolling_pca_scores(df: pd.DataFrame, tenors: list, window: int) -> pd.DataFrame:
    """Roll a PCA window across df. Returns PC1/PC2/PC3 scores."""
    records = []
    arr = df[tenors].values
    idx = df.index
    for i in range(window - 1, len(arr)):
        block = arr[i - window + 1 : i + 1]
        if np.isnan(block).any():
            records.append([np.nan] * 3)
            continue
        sc   = StandardScaler().fit_transform(block)
        p    = PCA(n_components=3).fit(sc)
        # score for the last (most recent) observation
        score = p.transform(sc[-1:, :])[0]
        records.append(score)
    out = pd.DataFrame(records, index=idx[window - 1:], columns=['PC1', 'PC2', 'PC3'])
    return out

print('Computing rolling PCA (slow — ~30s) ...')
roll_scores = rolling_pca_scores(raw, PCA_TENORS, ROLL_WIN)
print(f'Done. Shape: {roll_scores.shape}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
axes = axes.flatten()

pc_colors = {'PC1': '#58a6ff', 'PC2': '#3fb950', 'PC3': '#d29922'}

for ax, t in zip(axes, TRANSITIONS):
    nom = t['nomination']
    lo  = nom - pd.Timedelta(days=PRE_DAYS)
    hi  = t['start'] + pd.Timedelta(days=POST_DAYS)
    wd  = roll_scores.loc[lo:hi]
    x   = days_from_nomination(wd, nom)

    for pc, color in pc_colors.items():
        ax.plot(x, wd[pc], color=color, lw=1.5, label=pc)

    limbo_days = (t['start'] - nom).days
    ax.axvspan(0, limbo_days, alpha=0.12, color='#d29922')
    ax.axvline(0,          color='#d29922', lw=1.2, ls='--', alpha=0.8)
    ax.axvline(limbo_days, color='#3fb950', lw=1.2, ls='--', alpha=0.8)
    ax.axhline(0, color='#8b949e', lw=0.7, ls=':')

    ax.set_title(f'{t["label"]} — rolling PCA scores ({ROLL_WIN}d window)')
    ax.set_xlabel('Days from nomination')
    ax.set_ylabel('Score')
    ax.legend(fontsize=8)
    ax.grid(True)

fig.suptitle(
    f'Rolling PCA scores ({ROLL_WIN}-day window) around Fed Chair transitions\n'
    'PC1=level, PC2=slope, PC3=curvature  |  shaded=limbo  |  green dash=start',
    y=1.01
)
plt.tight_layout()
plt.show()

---
## 6 · Summary Statistics by Phase

In [ ]:
def phase_stats(df_yield, df_rvol, t, tenors=None):
    if tenors is None:
        tenors = ['2Y', '5Y', '10Y', '30Y']
    nom   = t['nomination']
    start = t['start']

    phases = {
        'pre':   (nom - pd.Timedelta(days=PRE_DAYS), nom - pd.Timedelta(days=1)),
        'limbo': (nom, start - pd.Timedelta(days=1)),
        'post':  (start, start + pd.Timedelta(days=POST_DAYS)),
    }

    rows = []
    for phase, (lo, hi) in phases.items():
        y  = df_yield.loc[lo:hi, tenors]
        rv = df_rvol.loc[lo:hi, tenors]
        chg = (y.iloc[-1] - y.iloc[0]) * 100  # bps
        avg_vol = rv.mean() 
        for tenor in tenors:
            rows.append({
                'transition': t['label'],
                'phase':      phase,
                'tenor':      tenor,
                'yield_chg_bps': round(chg[tenor], 1),
                'avg_rvol_bpd':  round(avg_vol[tenor], 2),
            })
    return rows

all_rows = []
for t in TRANSITIONS:
    all_rows.extend(phase_stats(raw, rvol, t))

summary = pd.DataFrame(all_rows)

# Pivot: tenor × phase for yield changes
pivot_chg = summary.pivot_table(
    index=['transition', 'tenor'], columns='phase', values='yield_chg_bps'
)[['pre', 'limbo', 'post']]

print('=== Yield change (bps) by phase ===')
print(pivot_chg.to_string())
print()

pivot_vol = summary.pivot_table(
    index=['transition', 'tenor'], columns='phase', values='avg_rvol_bpd'
)[['pre', 'limbo', 'post']]

print('=== Avg realised vol (bps/day) by phase ===')
print(pivot_vol.to_string())

In [ ]:
# Heatmap — yield change by phase
import matplotlib.colors as mcolors

def phase_heatmap(pivot, title, cmap='RdYlGn'):
    fig, ax = plt.subplots(figsize=(10, len(pivot) * 0.35 + 1.5))
    data = pivot.values.astype(float)
    vmax = np.nanpercentile(np.abs(data), 95)
    im = ax.imshow(data, cmap=cmap, aspect='auto', vmin=-vmax, vmax=vmax)

    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([' / '.join(map(str, idx)) for idx in pivot.index], fontsize=8)

    for i in range(data.shape[0]):
        for j in range(data.shape[1]):
            v = data[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.1f}', ha='center', va='center', fontsize=7,
                        color='black' if abs(v) < vmax * 0.5 else 'white')

    plt.colorbar(im, ax=ax, shrink=0.5)
    ax.set_title(title)
    plt.tight_layout()
    plt.show()

phase_heatmap(pivot_chg, 'Yield change (bps) by transition, tenor, and phase')
phase_heatmap(pivot_vol, 'Avg realised vol (bps/day) by transition, tenor, and phase',
              cmap='YlOrRd')

---
## 7 · Cross-transition Overlay

All transitions on the same axis, aligned to nomination day = 0. Best for spotting consistent patterns.

In [ ]:
OVERLAY_TENOR = '10Y'

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
series_map = [
    (axes[0], raw[OVERLAY_TENOR], 'Yield level (indexed to 0 at nomination)', True),
    (axes[1], rvol[OVERLAY_TENOR], '21d realised vol (bps/day)', False),
    (axes[2], roll_scores['PC2'],  f'Rolling PC2 (slope score, {ROLL_WIN}d)', False),
]

for ax, series, ylabel, index_to_zero in series_map:
    for i, t in enumerate(TRANSITIONS):
        nom = t['nomination']
        lo  = nom - pd.Timedelta(days=PRE_DAYS)
        hi  = t['start'] + pd.Timedelta(days=POST_DAYS)
        wd  = series.loc[lo:hi].dropna()
        x   = (wd.index - nom).days
        y   = wd.values
        if index_to_zero:
            base_val = series.loc[:nom].iloc[-1]
            y = (y - base_val) * 100
        ax.plot(x, y, color=PALETTE[i], lw=1.5, label=t['label'], alpha=0.85)

    ax.axvline(0, color='#d29922', lw=1.2, ls='--', alpha=0.9, label='Nomination')
    ax.axhline(0, color='#8b949e', lw=0.7, ls=':')
    ax.set_xlabel('Days from nomination')
    ax.set_ylabel(ylabel)
    ax.set_title(f'{OVERLAY_TENOR} — {ylabel.split("(")[0].strip()}')
    ax.legend(fontsize=7)
    ax.grid(True)

fig.suptitle(f'Cross-transition overlay  |  {OVERLAY_TENOR}  |  nomination = day 0', y=1.02)
plt.tight_layout()
plt.show()